# 🎯 Fase 2: Clasificación de Fallas (XGBoost) + Explicabilidad (SHAP)

En este notebook tomaremos el dataset enriquecido con las nuevas variables del notebook anterior. Nuestro objetivo es entrenar un clasificador que no solo nos diga **SI** va a fallar, sino **QUÉ TIPO** de falla va a ocurrir (`HDF`, `PWF`, `OSF`, etc.).

Además, usaremos **SHAP** para agregar explicabilidad: esto nos permitirá justificar cada predicción al operador de la planta (ej: 'Falla por sobrecalentamiento inminente debido a baja disipación térmica').

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import shap
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

plt.style.use('dark_background')
shap.initjs()

## 1. Carga de Datos y Preparación

In [ ]:
# Cargamos el dataset con las features derivadas
df = pd.read_csv('../data/ai4i2020_fe.csv')

# Quitamos Random Failures (RNF) ya que por definición, no se pueden predecir por telemetría
df = df[df['RNF'] == 0]

# Features que usaremos (X)
features = ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 
            'Torque [Nm]', 'Tool wear [min]', 'delta_temp', 'power_kw', 'wear_torque']
X = df[features]

# Crearemos una variable objetivo Multi-Clase (y)
# 0: Normal, 1: TWF, 2: HDF, 3: PWF, 4: OSF
def map_failure(row):
    if row['TWF'] == 1: return 1
    if row['HDF'] == 1: return 2
    if row['PWF'] == 1: return 3
    if row['OSF'] == 1: return 4
    return 0

y = df.apply(map_failure, axis=1)

print("Distribución de clases:")
print(y.value_counts())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Set de entrenamiento: {X_train.shape}")

## 2. Entrenamiento del Modelo (XGBoost)
Usaremos `scale_pos_weight` implícito manejando los pesos de clase para contrarrestar el fuerte desbalance.

In [ ]:
from sklearn.utils.class_weight import compute_sample_weight

sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

xgb_model = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=5,
    max_depth=5,
    learning_rate=0.1,
    n_estimators=100,
    random_state=42
)

xgb_model.fit(X_train, y_train, sample_weight=sample_weights)
print("Entrenamiento completado.")

## 3. Evaluación del Modelo

In [ ]:
y_pred = xgb_model.predict(X_test)

target_names = ['Normal', 'TWF (Wear)', 'HDF (Heat)', 'PWF (Power)', 'OSF (Overstrain)']
print(classification_report(y_test, y_pred, target_names=target_names))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(ax=ax, cmap='Blues')
plt.title('Matriz de Confusión')
plt.show()

## 4. Explicabilidad con SHAP
Vamos a explicar qué variables está utilizando el modelo para predecir las fallas.

In [ ]:
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

# SHAP Summary Plot para la clase 2 (HDF - Falla por calor)
print("Factores principales para detectar falla por disipación de calor (HDF):")
shap.summary_plot(shap_values[2], X_test, plot_type="bar")

In [ ]:
# Exportar modelo
joblib.dump(xgb_model, '../models/xgboost_classifier.pkl')
# Guardar explainer para la API
joblib.dump(explainer, '../models/shap_explainer.pkl')
print("Modelos guardados exitosamente en la carpeta 'models/'")